# Nominal associations

The association miner looks for pairwise implications between nominal
values: "when `segment = premium`, `shipping = express` in 95% of the
cases." It is the simplest rule detector in the package — one condition,
one outcome, nominal values only — and its statements are
`conditional_rules/if_then`, the same category PRISM writes to. That is
the point: the category says what was found, and finding it twice from
two angles strengthens one statement instead of duplicating two.

## What it mines

Eligible features are the nominal columns of the encoded matrix with at
most 16 distinct levels (standard budget). For every ordered pair of
values across two different features, the miner counts joint occurrences
with bitset masks — train rows in one mask, holdout rows in another. The
holdout is one fifth of the sampled rows, chosen by a stable hash of the
original row id, and it is *not* shown to the rule during discovery: a
rule qualifies on the train split (support ≥ 30, conditional share ≥
0.75, lift ≥ 1.5) and is then re-counted on the holdout, which is the
evidence the finding carries.

Denominators are pair-complete: rows where either side is missing simply
do not participate in that pair's counts. At most 20 findings survive on
`standard` (40 on `deep`), ranked deterministically.

In [ ]:
import random

import polars as pl

import onceover

rng = random.Random(23)
n = 4_000
plan = rng.choices(["basic", "pro"], weights=[60, 40], k=n)
support = [
    "phone" if (rng.random() < 0.9 if p == "pro" else rng.random() < 0.2)
    else "chat"
    for p in plan
]
df = pl.DataFrame({"plan": plan, "support": support})

report = onceover.profile(df)
rules = [f for f in report.findings if f["category"] == "conditional_rules"
         and any(e["method"]["detector"] == "association_rules" for e in f["evidence"])]
for f in rules:
    print(f["claim"])
    for e in f["evidence"]:
        if e["method"]["detector"] == "association_rules":
            m = e["metrics"]
            print(f"   train: n={m['train_n']} confidence={m['train_confidence']:.3f} lift={m['train_lift']:.2f} | "
                  f"holdout: n={m['holdout_n']} confidence={m['holdout_confidence']:.3f} lift={m['holdout_lift']:.2f}")

## Consolidation with rules

When PRISM and the association miner discover the same condition–outcome
pair — which is common, because a strong two-level implication is exactly
what both look for — the report shows one finding. The consolidation key
is the canonical statement: category, typed conditions, typed outcome.
The evidence list then contains the association miner's train/holdout
record, PRISM's discovery record, and the shared evaluator's partition
record, each with its own scope and validation label. A rule found by
three methods is a better-supported statement, not a tripled finding.

Different statements stay different: `plan = pro ⇒ support = phone` and
`support = phone ⇒ plan = pro` are separate claims with separate
evidence, because association is not symmetric.

## Limits, stated

The miner is bounded to *pairs* of nominal values: no three-item
conditions, no self-sufficiency correction against background knowledge
(Webb's filter), no statistical significance testing — the holdout
re-count is a consistency check, not an inferential guarantee. Lift is
measured against the outcome's rate on the same rows, so a rare outcome
with a modest conditional rate can look dramatic; read the joint count
before believing the lift. Wider itemsets and self-sufficiency filtering
are future directions ([](../appendix/directions.ipynb)).